# Cosmos 3 Edge を Colab で動かす

NVIDIA の 4B パラメータの世界モデル [nvidia/Cosmos3-Edge](https://huggingface.co/nvidia/Cosmos3-Edge) を 2 通りに試します。

- **Part A 推論 (Reasoner)**: 画像・動画を見て、文章で答える。Transformers で動かす
- **Part B 生成 (Generator)**: 1 枚の画像から、その先の動画を作る。Diffusers で動かす

**ランタイム**: 「ランタイム → ランタイムのタイプを変更」で **L4** か **A100** の GPU を選んでください。
T4 は対象外です (Ampere 以降の GPU と bf16 が前提)。

Part A と Part B は別々に動かせます。GPU メモリが足りないときは、Part A のあと「メモリを空ける」セルを実行してから Part B に進んでください。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
major, _ = torch.cuda.get_device_capability()
assert major >= 8, "Ampere 以降の GPU (L4 / A100 など) を選んでください。T4 では動きません。"

In [ ]:
# Transformers は Cosmos3 Edge に対応済みのリリースを入れる。Diffusers は main ブランチが必要
!pip install -q -U "transformers>=5.18" accelerate av imageio imageio-ffmpeg huggingface_hub
!pip install -q "diffusers @ git+https://github.com/huggingface/diffusers.git" cosmos_guardrail

In [ ]:
# モデルカードのお手本の入力 (画像・プロンプト) を取ってくる
from huggingface_hub import snapshot_download
repo = snapshot_download("nvidia/Cosmos3-Edge", allow_patterns=["assets/example_*", "assets/negative_prompt.json"])
print(repo)

## Part A 推論: 画像を見て作業計画を立てる

Reasoner の部分は Transformers に入っています (`AutoModelForImageTextToText`)。
既定では考えている過程 (`<think>` … `</think>`) を出してから答えます。

In [ ]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

model_id = "nvidia/Cosmos3-Edge"
model = AutoModelForImageTextToText.from_pretrained(model_id, torch_dtype=torch.bfloat16, device_map="auto")
processor = AutoProcessor.from_pretrained(model_id)

In [ ]:
from IPython.display import Image, display

def ask(content, max_new_tokens=1024, thinking=True):
    messages = [{"role": "user", "content": content}]
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=thinking,
    ).to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens)
    text = processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    if "</think>" in text:
        think, answer = text.split("</think>", 1)
        return think.strip(), answer.strip()
    return "", text.strip()

img = f"{repo}/assets/example_reasoning_input.png"
display(Image(img, width=480))

think, answer = ask([
    {"type": "image", "path": img},
    {"type": "text", "text": "The task is to put flower into the red bottle. Generate a plan consisting of subtasks for accomplish the task."},
])
print("--- 考えた過程 ---\n", think[:1500])
print("\n--- 答え ---\n", answer)

### 自分の画像で試す

下のセルを実行するとアップロード欄が出ます。質問は日本語でも通りますが、英語のほうが安定します。
`thinking=False` にすると考える過程を省いて速く答えます。

In [ ]:
from google.colab import files
up = files.upload()
my_img = next(iter(up))
display(Image(my_img, width=480))
_, answer = ask([
    {"type": "image", "path": my_img},
    {"type": "text", "text": "What objects are on the table, and where would a robot gripper need to move to pick up the nearest one?"},
], thinking=False)
print(answer)

### 動画で試す

動画も入力できます (4 fps くらいが推奨)。お手本にある車載カメラの動画で、次に何が起きそうかを聞きます。

In [ ]:
video = f"{repo}/assets/example_action_id_av_0_input.mp4"
_, answer = ask([
    {"type": "video", "path": video},
    {"type": "text", "text": "Describe the driving scene. What is the ego vehicle doing, and what should it watch out for next?"},
], thinking=False)
print(answer)

## メモリを空ける (Part B の前に)

In [ ]:
import gc
del model, processor
gc.collect(); torch.cuda.empty_cache()
print(f"{torch.cuda.memory_allocated()/1e9:.1f} GB 使用中")

## Part B 生成: 1 枚の画像から動画を作る

Diffusers の `Cosmos3OmniPipeline` を使います。プロンプトは文章ではなく、決まった形の JSON (プロンプトの「アップサンプリング」済み) が推奨です。
ここではお手本の JSON をそのまま使います。

お手本は 121 フレームですが、Colab で早く終わるよう 61 フレーム (24fps で約 2.5 秒) にしています。対応範囲は 50〜150 フレームです。
目安: A100 で数分、L4 ではその数倍かかります。

In [ ]:
import json
from pathlib import Path
import torch
from diffusers import Cosmos3OmniPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from diffusers.utils import export_to_video, load_image

pipe = Cosmos3OmniPipeline.from_pretrained(
    "nvidia/Cosmos3-Edge",
    torch_dtype=torch.bfloat16,
    enable_safety_checker=True,
)
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if total_gb < 40:
    pipe.enable_model_cpu_offload()  # L4 など: 使う部品だけ GPU に載せる
else:
    pipe.to("cuda")
pipe.scheduler = UniPCMultistepScheduler.from_config(
    pipe.scheduler.config, flow_shift=12.0, use_karras_sigmas=False
)

In [ ]:
json_prompt = json.loads(Path(f"{repo}/assets/example_i2v_prompt.json").read_text())
negative_prompt = json.loads(Path(f"{repo}/assets/negative_prompt.json").read_text())
first_frame = load_image(f"{repo}/assets/example_i2v_input.jpg")
display(first_frame.resize((416, 240)))
print(json.dumps(json_prompt, indent=2, ensure_ascii=False)[:1500])

In [ ]:
result = pipe(
    prompt=json.dumps(json_prompt),
    negative_prompt=json.dumps(negative_prompt),
    image=first_frame,
    num_frames=61,
    height=480,
    width=832,
    fps=24.0,
    num_inference_steps=20,
    guidance_scale=6.0,
    generator=torch.Generator(device="cuda").manual_seed(0),
    add_resolution_template=False,
    add_duration_template=False,
)
export_to_video(result.video, "i2v.mp4", fps=24, macro_block_size=1)

In [ ]:
from IPython.display import Video
Video("i2v.mp4", embed=True, width=832)

## Part C 自分の画像 + 指示文から動画を作る

流れは 3 段です。

1. 画像をアップロードし、どう動いてほしいかを指示文で書く (日本語で OK)
2. **Reasoner が画像を見て、指示文をお手本と同じ形の JSON プロンプトに書き直す** (プロンプトのアップサンプリングを Colab の中で済ませる。API キー不要)
3. Generator がその JSON と画像から動画を作る

GPU メモリを空けるため、2 と 3 ではモデルを入れ替えます。Part A・B を飛ばして、インストールとお手本取得のセルのあとにここから始めても動きます。

In [ ]:
# @title 1. 画像と指示文
import math
from pathlib import Path
from google.colab import files
from PIL import Image as PILImage, ImageOps
from IPython.display import display

instruction = "ロボットアームが赤いコップをつかんで、皿の上に置く"  # @param {type:"string"}
num_frames = 61  # @param {type:"slider", min:53, max:149, step:4}
FPS = 24

up = files.upload()
src_path = next(iter(up))

# 480p で対応している比率のうち、画像に一番近いものに合わせて中央を切り抜く (幅, 高さ)
SIZES = {"16,9": (832, 480), "4,3": (736, 544), "1,1": (640, 640), "3,4": (544, 736), "9,16": (480, 832)}
img = ImageOps.exif_transpose(PILImage.open(src_path)).convert("RGB")
aspect = min(SIZES, key=lambda k: abs(math.log(SIZES[k][0] / SIZES[k][1] / (img.width / img.height))))
W, H = SIZES[aspect]
first_frame = ImageOps.fit(img, (W, H), PILImage.BICUBIC)
first_frame.save("first_frame.png")
print(f"{W}x{H} ({aspect.replace(',', ':')}), {num_frames} フレーム = {num_frames / FPS:.1f} 秒")
display(first_frame.resize((W // 2, H // 2)))

In [ ]:
# @title 2. Reasoner に JSON プロンプトを書かせる
import gc, json, torch
from transformers import AutoModelForImageTextToText, AutoProcessor

for name in ("pipe", "model", "processor"):  # 前のモデルを GPU から降ろす
    globals().pop(name, None)
gc.collect(); torch.cuda.empty_cache()

model = AutoModelForImageTextToText.from_pretrained("nvidia/Cosmos3-Edge", torch_dtype=torch.bfloat16, device_map="auto")
processor = AutoProcessor.from_pretrained("nvidia/Cosmos3-Edge")

example = json.loads(Path(f"{repo}/assets/example_i2v_prompt.json").read_text())
duration = num_frames / FPS
request = f"""You write prompts for an image-to-video model. The attached image is the first frame of the video.
What should happen in the video: {instruction}
Video length: {duration:.1f} seconds.

Describe this video as one JSON object with exactly the same keys as the example below.
Describe what is really in the image (subjects, background, lighting, camera), then how the scene changes
according to the instruction, with timestamps in "actions" and "segments". Use "" for keys that do not apply.
Write every value in English. Output only the JSON.

Example:
{json.dumps(example, ensure_ascii=False)}"""

messages = [{"role": "user", "content": [
    {"type": "image", "path": "first_frame.png"},
    {"type": "text", "text": request},
]}]
inputs = processor.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_dict=True,
    return_tensors="pt", enable_thinking=False,
).to(model.device)
out = model.generate(**inputs, max_new_tokens=3072)
text = processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

try:
    json_prompt = json.loads(text[text.find("{"): text.rfind("}") + 1])
except json.JSONDecodeError:
    # 崩れた JSON が返ってきたら、指示文だけを入れた最小の形で代用する
    print("JSON として読めなかったので、指示文だけで組みます\n", text[:500])
    json_prompt = {k: ([] if isinstance(v, list) else {} if isinstance(v, dict) else "") for k, v in example.items()}
    json_prompt["temporal_caption"] = instruction
    json_prompt["actions"] = [{"time": f"0:00-0:{math.ceil(duration):02d}", "description": instruction}]

# 解像度と長さは実際に作る値に揃える
json_prompt.update({"resolution": {"W": W, "H": H}, "aspect_ratio": aspect,
                    "duration": f"{round(duration)}s", "fps": FPS})
Path("my_prompt.json").write_text(json.dumps(json_prompt, indent=2, ensure_ascii=False))
print(json.dumps(json_prompt, indent=2, ensure_ascii=False)[:3000])

del model, processor
gc.collect(); torch.cuda.empty_cache()

JSON を手で直したいときは、左のファイル欄で `my_prompt.json` をダブルクリックして編集・保存してから次のセルを実行してください (次のセルはこのファイルを読み直します)。
動きが弱いときは `actions` と `segments` の説明を具体的にすると効きます。

In [ ]:
# @title 3. 動画を作る
import json, torch
from diffusers import Cosmos3OmniPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from diffusers.utils import export_to_video
from IPython.display import Video

seed = 0  # @param {type:"integer"}

if "pipe" not in globals():
    pipe = Cosmos3OmniPipeline.from_pretrained("nvidia/Cosmos3-Edge", torch_dtype=torch.bfloat16, enable_safety_checker=True)
    if torch.cuda.get_device_properties(0).total_memory / 1e9 < 40:
        pipe.enable_model_cpu_offload()
    else:
        pipe.to("cuda")
    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=12.0, use_karras_sigmas=False)

json_prompt = json.loads(Path("my_prompt.json").read_text())
negative_prompt = json.loads(Path(f"{repo}/assets/negative_prompt.json").read_text())

result = pipe(
    prompt=json.dumps(json_prompt),
    negative_prompt=json.dumps(negative_prompt),
    image=PILImage.open("first_frame.png"),
    num_frames=num_frames,
    height=H,
    width=W,
    fps=float(FPS),
    num_inference_steps=20,
    guidance_scale=6.0,
    generator=torch.Generator(device="cuda").manual_seed(seed),
    add_resolution_template=False,
    add_duration_template=False,
)
out_path = f"my_video_seed{seed}.mp4"
export_to_video(result.video, out_path, fps=FPS, macro_block_size=1)
display(Video(out_path, embed=True, width=min(W, 832)))
# files.download(out_path)  # 手元に保存したいとき

- 同じ JSON で `seed` だけ変えて何本か作ると、当たりが出やすくなります (2 回目以降はモデルの読み込みが省かれます)
- 指示文を変えたいときは 1 → 2 → 3 の順にやり直します
- 真っ黒・空の動画になったときは安全フィルタ (`cosmos_guardrail`) に止められた可能性があります